# 06 · Long-context stress grid (v18)

Extracts the experiment design from `DIINA_v18_LONG_CONTEXT_STRESS.py`: lengths 10/50/100/200 crossed with noise 0/0.1/0.2, 200 generated examples per split. The script's reported accuracy requires actually training its PyTorch models; here we provide an explicitly **illustrative statistical proxy**, never label it measured accuracy.

In [ ]:
from pathlib import Path
import json, math, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def find_project():
    candidates=[Path.cwd(), Path.cwd()/"project_inputs", Path.cwd().parent/"project_inputs", Path("/mnt/data/diina_reproducibility/project_inputs"), Path("/mnt/data")]
    for p in candidates:
        if (p/"raw_results.csv").exists(): return p
    raise FileNotFoundError("Place notebook with project_inputs/ or set cwd to extracted project directory")
PROJECT=find_project()
df=pd.read_csv(PROJECT/"raw_results.csv")
summary=json.loads((PROJECT/"summary.json").read_text())
print("Project files:",PROJECT,"| telemetry shape:",df.shape,"| columns:",list(df.columns))
assert len(df)==1500 and set(df.columns)=={"seed","step","loss","fast_usage","slow_usage","entropy"}

In [ ]:
src=(PROJECT/"DIINA_v18_LONG_CONTEXT_STRESS.py").read_text()
for pattern in ["lengths=", "noises=", "range(200)", "TransformerModel", "DIINA"]: print(pattern,pattern in src)
lengths=[10,50,100,200]; noises=[0.0,0.1,0.2]
grid=pd.MultiIndex.from_product([lengths,noises],names=["length","noise"]).to_frame(index=False)
# Transparent illustrative expected-error curve, not benchmark output.
grid["proxy_error"]=np.minimum(1,grid.noise+0.015*np.log1p(grid.length/10))
grid["proxy_accuracy"]=1-grid.proxy_error
display(grid)
assert len(grid)==12 and set(grid.length)==set(lengths) and set(grid.noise)==set(noises)

In [ ]:
try:
    import ipywidgets as widgets
    @widgets.interact(length=widgets.SelectionSlider(options=lengths,value=50),noise=widgets.SelectionSlider(options=noises,value=.1))
    def inspect_condition(length=50,noise=.1):
        row=grid.query("length==@length and noise==@noise").iloc[0]
        print(f"Condition length={length}, noise={noise}; illustrative proxy accuracy={row.proxy_accuracy:.3f} (not measured)")
except ImportError: print("Install ipywidgets for optional interactive controls")
fig,ax=plt.subplots(figsize=(7,4))
for noise,g in grid.groupby("noise"): ax.plot(g.length,g.proxy_accuracy,marker="o",label=f"noise={noise}")
ax.set(xlabel="sequence length",ylabel="illustrative proxy accuracy",ylim=(0,1.02),title="Illustrative degradation model only"); ax.legend(); plt.show()

**Source caveat:** The supplied v18 file contains redacted/opaque tokens in its generator body; do not treat execution of that source as verified. The grid is faithfully ported from readable configuration, while all proxy values are pedagogical.